In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms, models

import albumentations as A

try:
    from efficientnet_pytorch import EfficientNet

    _use_external = True
except ImportError:
    _use_external = False
    EfficientNet = None




In [2]:
possible_paths = [
    "../input/en-b4-tta-calr-40/model_40.pth",
    "/kaggle/input/en-b4-tta-calr-40/model_40.pth",
    "./input/en-b4-tta-calr-40/model_40.pth",
]
model_path = next((p for p in possible_paths if os.path.isfile(p)), None)
sample_sub_path = "../input/cassava-leaf-disease-classification/sample_submission.csv"
test_images_path = "../input/cassava-leaf-disease-classification/test_images"

model_exists = model_path is not None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if _use_external:
    model = EfficientNet.from_name("efficientnet-b4", num_classes=5)
else:
    model = models.efficientnet_b4(pretrained=True)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 5)

model = model.to(device)

if model_exists:
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    print(f"Loaded checkpoint from {model_path}")
else:
    print("Checkpoint not found – will fine‑tune briefly on a larger training subset.")
    train_df = pd.read_csv("../input/cassava-leaf-disease-classification/train.csv")
    # use a larger random subset for better fine‑tuning
    train_subset = train_df.sample(n=5000, random_state=42).reset_index(drop=True)

    class CassavaDataset(torch.utils.data.Dataset):
        def __init__(self, df, img_dir, transform):
            self.df = df
            self.img_dir = img_dir
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img_path = os.path.join(self.img_dir, row["image_id"])
            img = np.array(Image.open(img_path).convert("RGB"))
            img = self.transform(image=img)["image"]
            img = torch.from_numpy(img).permute(2, 0, 1).float()
            label = torch.tensor(row["label"], dtype=torch.long)
            return img, label

    train_aug = A.Compose(
        [
            A.Resize(256, 256),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
                max_pixel_value=255.0,
            ),
        ]
    )

    train_dataset = CassavaDataset(
        train_subset,
        "../input/cassava-leaf-disease-classification/train_images",
        train_aug,
    )
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)  # slightly lower LR
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

    model.train()
    for epoch in range(5):  # increase epochs for a more effective lift
        epoch_loss = 0.0
        for imgs, lbls in train_loader:
            imgs = imgs.to(device)
            lbls = lbls.to(device)

            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, lbls)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * imgs.size(0)

        scheduler.step()
        print(
            f"Fine‑tune epoch {epoch+1}/5 – loss: {epoch_loss/len(train_loader.dataset):.4f}"
        )

    model.eval()




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B4_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B4_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:00<00:00, 110MB/s]


Checkpoint not found – will fine‑tune briefly on a larger training subset.
Fine‑tune epoch 1/5 – loss: 0.8653
Fine‑tune epoch 2/5 – loss: 0.5027
Fine‑tune epoch 3/5 – loss: 0.3145
Fine‑tune epoch 4/5 – loss: 0.2302
Fine‑tune epoch 5/5 – loss: 0.1618


In [3]:
sub_aug = A.Compose(
    [
        A.Resize(height=256, width=256),
        A.Transpose(p=0.8),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(p=0.8),
        A.HueSaturationValue(
            hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=0, p=0.5
        ),
        A.RandomBrightnessContrast(
            brightness_limit=(-0.1, 0.1), contrast_limit=(-0.1, 0.1), p=0.5
        ),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
            p=1.0,
        ),
        # reduce aggressiveness of dropout to keep more signal
        A.CoarseDropout(max_holes=20, max_height=10, max_width=10, p=0.3),
    ],
    p=1.0,
)




/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_55/677827892.py:21: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=20, max_height=10, max_width=10, p=0.3),


In [4]:
sample_sub = pd.read_csv(sample_sub_path)

predictions = []

with torch.no_grad():
    for _, sample_row in sample_sub.iterrows():
        image_path = os.path.join(test_images_path, sample_row.image_id)
        if not os.path.isfile(image_path):
            pred_label = 0
            predictions.append([sample_row.image_id, pred_label])
            continue

        tta_sum = torch.zeros(5, device=device)  # accumulate logits over TTA
        for _ in range(5):  # 5 TTA augmentations
            img = np.array(Image.open(image_path).convert("RGB"))
            img = sub_aug(image=img)["image"]
            img_tensor = torch.from_numpy(img).permute(2, 0, 1).float().to(device)
            outputs = model(img_tensor.unsqueeze(0)).squeeze(0)  # logits shape (5,)
            tta_sum += outputs

        avg_logits = tta_sum / 5.0
        pred_label = torch.argmax(avg_logits).item()
        predictions.append([sample_row.image_id, pred_label])

sub_df = pd.DataFrame(predictions, columns=["image_id", "label"])
sub_df.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")
print(sub_df.head())

Submission saved to submission.csv
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
